# SIDIX — fine-tune Qwen2.5-7B-Instruct (QLoRA) di Kaggle

Versi kurasi repo (tanpa output besar).

**Fix path:** Auto-detect JSONL di `/kaggle/input/sidix-sft-dataset/` (path standar Kaggle, tanpa prefix `datasets/mighan/`).

Dataset: sidix-sft-dataset → finetune_sft.jsonl  
Output: /kaggle/working/sidix-lora-adapter/ → zip untuk unduh.

In [ ]:
!pip install -q peft trl bitsandbytes accelerate datasets

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import glob
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ── Auto-detect path dataset (Kaggle strips owner prefix) ──────────────────
# Kaggle standard path: /kaggle/input/<dataset-slug>/<file>
# NOT: /kaggle/input/datasets/<owner>/<dataset-slug>/<file>
_candidates = [
    "/kaggle/input/sidix-sft-dataset",
    "/kaggle/input/datasets/mighan/sidix-sft-dataset",
]
_jsonl_files = []
for _d in _candidates:
    _jsonl_files = sorted(glob.glob(f"{_d}/*.jsonl"))
    if _jsonl_files:
        print(f"Dataset dir: {_d}")
        break

if not _jsonl_files:
    # Last resort: scan all /kaggle/input/
    _jsonl_files = sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True))
    print(f"Fallback scan found: {_jsonl_files}")

if not _jsonl_files:
    raise FileNotFoundError(
        "finetune_sft.jsonl tidak ditemukan di /kaggle/input/. "
        "Pastikan dataset 'sidix-sft-dataset' sudah di-attach ke notebook ini."
    )

DATASET_PATH = _jsonl_files[0]
print(f"Using: {DATASET_PATH}")

OUTPUT_DIR  = "/kaggle/working/sidix-qwen2.5-7b-lora"
ADAPTER_DIR = "/kaggle/working/sidix-lora-adapter"
MODEL_ID    = "Qwen/Qwen2.5-7B-Instruct"

# ── Load dataset ───────────────────────────────────────────────────────────
records = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
dataset = Dataset.from_list(records)
print(f"Dataset loaded: {len(dataset)} samples")

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

# ── Load model 4-bit ───────────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
print("Model loaded OK")

# ── LoRA ───────────────────────────────────────────────────────────────────
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Format ChatML ──────────────────────────────────────────────────────────
def format_chatml(example):
    text = ""
    for msg in example["messages"]:
        text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    text += "<|im_start|>assistant\n"
    return {"text": text}

train_dataset = train_dataset.map(format_chatml)
eval_dataset  = eval_dataset.map(format_chatml)

# ── Training ───────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

# ── Save adapter ───────────────────────────────────────────────────────────
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
import shutil
from pathlib import Path

shutil.make_archive(
    "/kaggle/working/sidix-lora-adapter",
    "zip",
    root_dir="/kaggle/working",
    base_dir="sidix-lora-adapter",
)
size_mb = Path("/kaggle/working/sidix-lora-adapter.zip").stat().st_size / 1024**2
print(f"OK: sidix-lora-adapter.zip ({size_mb:.1f} MiB)")